In [5]:
!git config --global user.name "chamudiR"
!git config --global user.email "chamudiransika@gmail.com"


In [6]:
!git clone https://github.com/chamudiR/vggt.git

fatal: destination path 'vggt' already exists and is not an empty directory.


In [7]:
%cd vggt


/content/vggt/vggt


In [ ]:
# Cell 1: Upload images from your computer
from google.colab import files
import os

# Create images folder
os.makedirs('images', exist_ok=True)

print("Click 'Choose Files' and select your fern images (004.png to 018.png)")
uploaded = files.upload()

# Move to images folder
for filename in uploaded.keys():
    os.rename(filename, f'images/{filename}')

print(f"\n✓ Uploaded {len(uploaded)} images")
!ls images/


Click 'Choose Files' and select your fern images (004.png to 018.png)


Saving 000.png to 000.png
Saving 001.png to 001.png
Saving 002.png to 002.png
Saving 003.png to 003.png
Saving 004.png to 004.png
Saving 005.png to 005.png
Saving 006.png to 006.png
Saving 007.png to 007.png
Saving 008.png to 008.png
Saving 009.png to 009.png
Saving 010.png to 010.png
Saving 011.png to 011.png
Saving 012.png to 012.png
Saving 013.png to 013.png
Saving 014.png to 014.png
Saving 015.png to 015.png
Saving 016.png to 016.png
Saving 017.png to 017.png
Saving 018.png to 018.png
Saving 019.png to 019.png

✓ Uploaded 20 images
000.png  003.png  006.png  009.png  012.png  015.png  018.png
001.png  004.png  007.png  010.png  013.png  016.png  019.png
002.png  005.png  008.png  011.png  014.png  017.png


In [ ]:
# Cell 1: Fix NumPy and prepare environment
print("Fixing NumPy version incompatibility...")

# Uninstall conflicting packages
!pip uninstall numpy scipy -y -q

# Install correct NumPy version
!pip install numpy==1.26.4 -q

print("✓ NumPy 1.26.4 installed")
print("\n⚠️ IMPORTANT: Click 'Runtime' → 'Restart runtime' NOW!")
print("Then proceed to Cell 2 after restart")



Fixing NumPy version incompatibility...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hyperopt 0.2.7 requires scipy, which is not installed.
scikit-learn 1.6.1 requires scipy>=1.6.0, which is not installed.
fastai 2.8.4 requires scipy, which is not installed.
statsmodels 0.14.5 requires scipy!=1.9.2,>=1.8, which is not installed.
jaxlib 0.5.3 requires scipy>=1.11.1, which is not installed.
arviz 0.22.0 requires scipy>=1.11.0, which is not installed.
imbalanced-learn 0.14.0 requires scipy<2,>=1.11.4, which is not installed.
scikit-image 0.25.2 requires scipy>=1.11.4, which is not installed.
cuml-cu12 25.6.0 requires scipy>=1.8.0, which is not installed.
scs 3.2.8 requires scipy, which is not installed.
treelite 4.4.1 requires scipy, which is not installed.
albumentations 2.0.8 requires scipy>=1.10.0, which is not installed.
pytensor 2.31.7 requires scipy<2,>=1

In [ ]:
# Cell 2: Install VGGT (run AFTER restarting runtime)
print("Installing VGGT...")

# Verify NumPy version first
import numpy as np
print(f"NumPy version: {np.__version__}")

if np.__version__ != "1.26.4":
    print("ERROR: NumPy version is wrong! Re-run Cell 1 and restart runtime.")
    exit()

# Install VGGT without upgrading NumPy
!pip install --no-deps git+https://github.com/facebookresearch/vggt.git -q
!pip install huggingface_hub pillow torch torchvision -q

print("✓ VGGT installed")
print("\nVerifying installation:")
import torch
from vggt.models.vggt import VGGT
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}")
print(f"  VGGT: OK")


Installing VGGT...
NumPy version: 1.26.4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✓ VGGT installed

Verifying installation:
  PyTorch: 2.8.0+cu126
  CUDA: True
  VGGT: OK


In [ ]:
# Cell 3: VGGT Inference - COMPLETE FIXED VERSION
import torch
import numpy as np
from pathlib import Path
from glob import glob
import time

print("="*70)
print("VGGT 3D RECONSTRUCTION - Google Colab GPU")
print("="*70)

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✓ Device: {device}")
if device == "cuda":
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Get images
image_paths = sorted(glob('images/*.png'))
print(f"\n✓ Found {len(image_paths)} images")

# Load model
print("\n" + "="*70)
print("LOADING MODEL")
print("="*70)
print("Downloading from HuggingFace (~4.8GB)...")

start = time.time()

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

model = VGGT.from_pretrained("facebook/VGGT-1B")
model = model.to(device).eval()

print(f"✓ Model loaded in {time.time()-start:.1f}s")

# Load images
print("\n" + "="*70)
print("LOADING IMAGES")
print("="*70)

images = load_and_preprocess_images(image_paths).to(device)
print(f"✓ Images: {images.shape}")

# Inference
print("\n" + "="*70)
print("RUNNING INFERENCE")
print("="*70)

inference_start = time.time()

with torch.no_grad():
    batch = images.unsqueeze(0)

    print("[1/4] Aggregator...", end=" ", flush=True)
    t = time.time()
    tokens, ps_idx = model.aggregator(batch)
    print(f"✓ {time.time()-t:.1f}s")

    print("[2/4] Camera head...", end=" ", flush=True)
    t = time.time()
    pose_enc = model.camera_head(tokens)[-1]
    extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, batch.shape[-2:])
    print(f"✓ {time.time()-t:.1f}s")

    print("[3/4] Depth head...", end=" ", flush=True)
    t = time.time()
    depth_map, depth_conf = model.depth_head(tokens, batch, ps_idx)
    print(f"✓ {time.time()-t:.1f}s")

    print("[4/4] Unprojecting...", end=" ", flush=True)
    t = time.time()
    point_map = unproject_depth_map_to_point_map(
        depth_map.squeeze(0),
        extrinsic.squeeze(0),
        intrinsic.squeeze(0)
    )
    print(f"✓ {time.time()-t:.1f}s")

print(f"\n✓ Total inference: {time.time()-inference_start:.1f}s")

# FIXED SAVING SECTION
print("\n" + "="*70)
print("SAVING POINT CLOUD")
print("="*70)

# Handle both torch tensors and numpy arrays
if isinstance(point_map, torch.Tensor):
    points = point_map.cpu().numpy()
else:
    points = point_map

if isinstance(images, torch.Tensor):
    colors = images.cpu().numpy()
else:
    colors = images

# Reshape
N, H, W, _ = points.shape
points = points.reshape(-1, 3)
colors = colors.transpose(0, 2, 3, 1).reshape(-1, 3)

# Filter valid points
print(f"Total points before filtering: {len(points):,}")
valid = ~np.isnan(points).any(axis=1) & ~np.isinf(points).any(axis=1)
valid &= (np.linalg.norm(points, axis=1) < 100)

points = points[valid]
colors = (colors[valid] * 255).clip(0, 255).astype(np.uint8)

print(f"Valid points after filtering: {len(points):,}")

# Save PLY
filename = "vggt_reconstruction.ply"
print(f"Writing {filename}...")

with open(filename, 'w') as f:
    f.write("ply\n")
    f.write("format ascii 1.0\n")
    f.write(f"element vertex {len(points)}\n")
    f.write("property float x\n")
    f.write("property float y\n")
    f.write("property float z\n")
    f.write("property uchar red\n")
    f.write("property uchar green\n")
    f.write("property uchar blue\n")
    f.write("end_header\n")

    for i in range(len(points)):
        f.write(f"{points[i,0]:.6f} {points[i,1]:.6f} {points[i,2]:.6f} ")
        f.write(f"{colors[i,0]} {colors[i,1]} {colors[i,2]}\n")

print(f"✓ Saved {filename} ({len(points):,} points)")

# Save camera parameters
if isinstance(extrinsic, torch.Tensor):
    np.save("camera_extrinsics.npy", extrinsic.cpu().numpy())
    np.save("camera_intrinsics.npy", intrinsic.cpu().numpy())
    np.save("depth_maps.npy", depth_map.cpu().numpy())
else:
    np.save("camera_extrinsics.npy", extrinsic)
    np.save("camera_intrinsics.npy", intrinsic)
    np.save("depth_maps.npy", depth_map)

print("✓ Saved camera parameters")

print("\n" + "="*70)
print("✓✓✓ RECONSTRUCTION COMPLETED!")
print("="*70)
print(f"Images: {len(image_paths)}")
print(f"Points: {len(points):,}")
print(f"Output: {filename}")
print("="*70)

# Download files
from google.colab import files
print("\nDownloading files...")
files.download('vggt_reconstruction.ply')
files.download('camera_extrinsics.npy')
files.download('camera_intrinsics.npy')
files.download('depth_maps.npy')
print("✓ All files downloaded!")


VGGT 3D RECONSTRUCTION - Google Colab GPU

✓ Device: cuda
✓ GPU: Tesla T4
✓ VRAM: 15.8 GB

✓ Found 20 images

LOADING MODEL
✓ Model loaded in 34.3s

LOADING IMAGES
✓ Images: torch.Size([20, 3, 392, 518])

RUNNING INFERENCE
[1/4] Aggregator... ✓ 29.5s
[2/4] Camera head... ✓ 0.0s
[3/4] Depth head... ✓ 1.4s
[4/4] Unprojecting... ✓ 1.3s

✓ Total inference: 32.2s

SAVING POINT CLOUD
Total points before filtering: 4,061,120
Valid points after filtering: 4,061,120
Writing vggt_reconstruction.ply...
✓ Saved vggt_reconstruction.ply (4,061,120 points)
✓ Saved camera parameters

✓✓✓ RECONSTRUCTION COMPLETED!
Images: 20
Points: 4,061,120
Output: vggt_reconstruction.ply



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ All files downloaded!


In [ ]:
# Visualize point cloud in Colab
!pip install -q plotly

import plotly.graph_objects as go
import numpy as np

print("Loading point cloud for visualization...")

# If you already have the processed points and colors from previous cell
if 'points' in locals() and 'colors' in locals():
    pts = points
    cols = colors
else:
    # Load from PLY file
    print("Reading vggt.ply...")
    with open('vggt.ply', 'r') as f:
        lines = f.readlines()

    # Find header end
    header_end = lines.index('end_header\n') + 1

    # Parse points
    data = []
    for line in lines[header_end:]:
        vals = line.strip().split()
        data.append([float(vals[0]), float(vals[1]), float(vals[2]),
                     int(vals[3]), int(vals[4]), int(vals[5])])

    data = np.array(data)
    pts = data[:, :3]
    cols = data[:, 3:6].astype(int)

# Subsample for faster visualization (show every 10th point)
step = max(1, len(pts) // 50000)  # Limit to 50k points for performance
pts_sub = pts[::step]
cols_sub = cols[::step]

print(f"Visualizing {len(pts_sub):,} points (subsampled from {len(pts):,})...")

# Create RGB strings for plotly
rgb_colors = [f'rgb({c[0]},{c[1]},{c[2]})' for c in cols_sub]

# Create 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=pts_sub[:, 0],
    y=pts_sub[:, 1],
    z=pts_sub[:, 2],
    mode='markers',
    marker=dict(
        size=2,
        color=rgb_colors,
    )
)])

fig.update_layout(
    title=f"VGGT 3D Reconstruction ({len(pts):,} points)",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    width=900,
    height=700,
)

print("✓ Rendering interactive 3D visualization...")
fig.show()

print("\n✓ Done! Use mouse to rotate/zoom the point cloud")


Loading point cloud for visualization...
Visualizing 50,138 points (subsampled from 4,061,120)...
✓ Rendering interactive 3D visualization...



✓ Done! Use mouse to rotate/zoom the point cloud


In [9]:
!git fetch origin



In [11]:
!git checkout -b chamudiR
!git push -u origin chamudiR


Switched to a new branch 'chamudiR'
fatal: could not read Username for 'https://github.com': No such device or address
